In [5]:
import pandas as pd
import os

df = pd.read_csv("output/training_data.csv")
print("Top 5 rows:")
print(df.head())
print("\nNumber of rows:", df.shape[0])

Top 5 rows:
        A                           Player UUID  Code  \
0  150329  05b195d0-ea16-4d04-bc9b-8567821f271b    12   
1   94783  0ee37250-4bdb-4c84-94f0-afa10ac4a9af    10   
2   36939  107ff853-e67a-4551-abb0-0c6f29d55e44    21   
3   61776  2403603b-153c-4391-a800-f6c4e7536487    18   
4    3977  3003d693-61f9-4157-bf7f-51389503be0c    20   

                 Player Name        Web Name Player Team Name   season  \
0  Sokratis Papastathopoulos        Sokratis          Arsenal  2018-19   
1    Konstantinos Mavropanos      Mavropanos          Arsenal  2018-19   
2              Danny Welbeck         Welbeck          Arsenal  2018-19   
3         Henrikh Mkhitaryan      Mkhitaryan          Arsenal  2018-19   
4     Ainsley Maitland-Niles  Maitland-Niles          Arsenal  2018-19   

   Gameweek  Minutes Played  Goals Scored  ...  Team_Total_Points_GW  \
0         1              90             0  ...                    19   
1         1               0             0  ...          

In [6]:
# Load fixtures for a specific season to investigate
def load_fixtures_for_season(season, data_folder='data'):
    fixtures_path = os.path.join(data_folder, season, 'fixtures.csv')
    return pd.read_csv(fixtures_path)

In [13]:
# ## Step 1: Investigate the specific case (Sokratis, GW29, 2019-20)

# %%
# Filter for the specific problematic row
problematic_row = df[
    (df['Player Name'] == 'Sokratis Papastathopoulos') & 
    (df['season'] == '2019-20') & 
    (df['Gameweek'] == 29)
]

print("Training Data Row:")
print(problematic_row[['Player Name', 'Player Team Name', 'season', 'Gameweek', 
                       'Opponent Name', 'Opponent ID', 'Is Home', 'Opponent Difficulty']])
# Load fixtures for that season and gameweek
fixtures_2019_20 = load_fixtures_for_season('2019-20', data_folder='data')
gw29_fixtures = fixtures_2019_20[fixtures_2019_20['event'] == 29]

print("\nGameweek 29 Fixtures (2019-20):")
print(gw29_fixtures[['team_h', 'team_a', 'team_h_difficulty', 'team_a_difficulty']])

# %%
# Load teams data to map team IDs to names
teams_2019_20 = pd.read_csv('data/2019-20/teams.csv')
print("\nTeams mapping:")
print(teams_2019_20[['id', 'name']])

# %%
# Find Arsenal's team ID
arsenal_id = teams_2019_20[teams_2019_20['name'] == 'Arsenal']['id'].values[0]
print(f"\nArsenal Team ID: {arsenal_id}")

# Find Arsenal's match in GW29
arsenal_match_gw29 = gw29_fixtures[
    (gw29_fixtures['team_h'] == arsenal_id) | 
    (gw29_fixtures['team_a'] == arsenal_id)
]

print("\nArsenal's match in GW29:")
print(arsenal_match_gw29)

# Map team IDs to names
if not arsenal_match_gw29.empty:
    match = arsenal_match_gw29.iloc[0]
    home_team = teams_2019_20[teams_2019_20['id'] == match['team_h']]['name'].values[0]
    away_team = teams_2019_20[teams_2019_20['id'] == match['team_a']]['name'].values[0]
    print(f"\nActual Match: {home_team} vs {away_team}")
    print(f"Arsenal was: {'Home' if match['team_h'] == arsenal_id else 'Away'}")

Training Data Row:
                    Player Name Player Team Name   season  Gameweek  \
1894  Sokratis Papastathopoulos          Arsenal  2019-20        29   

     Opponent Name  Opponent ID  Is Home  Opponent Difficulty  
1894      Man City         19.0     True                  2.0  

Gameweek 29 Fixtures (2019-20):
     team_h  team_a  team_h_difficulty  team_a_difficulty
278      10       3                  2                  5
279       1      19                  2                  4
280       7      18                  2                  3
281      15      14                  2                  3
282      16      13                  2                  2
283      20       4                  2                  3
284       5      17                  3                  3
285       6       8                  2                  4
286      12      11                  4                  4
287       9       2                  2                  3

Teams mapping:
    id            name


In [15]:
# ## Step 2: Check how Opponent Name and Opponent ID are created in your training data
# This will help identify where the mismatch comes from
# Check if the issue is in how opponents are being assigned initially

print("\nLet's check if there are multiple entries with same issue:")
gw29_arsenal_players = df[
    (df['season'] == '2019-20') & 
    (df['Gameweek'] == 29) & 
    (df['Player Team Name'] == 'Arsenal')
]

print(f"\nAll Arsenal players in GW29:")
print(gw29_arsenal_players[['Player Name', 'Opponent Name', 'Opponent ID', 'Is Home']].head(10))




Let's check if there are multiple entries with same issue:

All Arsenal players in GW29:
                           Player Name Opponent Name  Opponent ID  Is Home
1894         Sokratis Papastathopoulos      Man City         19.0     True
1895           Konstantinos Mavropanos      Man City         19.0     True
1896                    Kieran Tierney      Man City         19.0     True
1897                    Joseph Willock      Man City         19.0     True
1898  Gabriel Teodoro Martinelli Silva      Man City         19.0     True
1899                Henrikh Mkhitaryan      Man City         19.0     True
1900                       Bukayo Saka      Man City         19.0     True
1901            Ainsley Maitland-Niles      Man City         19.0     True
1902               Alexandre Lacazette      Man City         19.0     True
1903                Tyreece John-Jules      Man City         19.0     True


In [16]:
# ## Step 3: Identify the root cause pattern

# Let's check if Opponent ID is correct but Opponent Name is wrong, or vice versa
if not problematic_row.empty:
    opponent_id = problematic_row['Opponent ID'].values[0]
    opponent_name_in_data = problematic_row['Opponent Name'].values[0]
    
    print(f"\nOpponent ID in training data: {opponent_id}")
    print(f"Opponent Name in training data: {opponent_name_in_data}")
    
    # Check what team this ID actually corresponds to
    actual_team = teams_2019_20[teams_2019_20['id'] == opponent_id]
    if not actual_team.empty:
        print(f"Team ID {opponent_id} actually corresponds to: {actual_team['name'].values[0]}")



Opponent ID in training data: 19.0
Opponent Name in training data: Man City
Team ID 19.0 actually corresponds to: West Ham


In [17]:
# ## Step 4: Fix the issue

# %%
def get_player_team_id(player_team_name, season, data_folder='data'):
    """Get team ID from team name for a specific season"""
    teams_path = os.path.join(data_folder, season, 'teams.csv')
    teams_df = pd.read_csv(teams_path)
    team = teams_df[teams_df['name'] == player_team_name]
    if not team.empty:
        return team['id'].values[0]
    return None

def fix_opponent_and_difficulty(df, data_folder='data'):
    """
    Rebuild Opponent Name, Opponent ID, and Opponent Difficulty from fixtures
    """
    fixed_opponent_ids = []
    fixed_opponent_names = []
    fixed_opponent_difficulties = []
    fixed_is_home = []
    
    match_dictionary = load_fixtures(data_folder)
    
    for idx, row in df.iterrows():
        season = row['season']
        gameweek = row['Gameweek']
        player_team_name = row['Player Team Name']
        
        try:
            # Get player's team ID
            teams_df = pd.read_csv(os.path.join(data_folder, season, 'teams.csv'))
            player_team = teams_df[teams_df['name'] == player_team_name]
            
            if player_team.empty:
                fixed_opponent_ids.append(None)
                fixed_opponent_names.append(None)
                fixed_opponent_difficulties.append(None)
                fixed_is_home.append(None)
                continue
                
            player_team_id = player_team['id'].values[0]
            
            # Find the match in fixtures
            gameweek_matches = match_dictionary[season][gameweek]
            
            opponent_id = None
            opponent_name = None
            difficulty = None
            is_home = None
            
            for match in gameweek_matches:
                if match['team_h'] == player_team_id:
                    # Player's team is home
                    is_home = True
                    opponent_id = match['team_a']
                    difficulty = match['team_h_difficulty']
                    break
                elif match['team_a'] == player_team_id:
                    # Player's team is away
                    is_home = False
                    opponent_id = match['team_h']
                    difficulty = match['team_a_difficulty']
                    break
            
            # Get opponent name from team ID
            if opponent_id is not None:
                opponent_team = teams_df[teams_df['id'] == opponent_id]
                if not opponent_team.empty:
                    opponent_name = opponent_team['name'].values[0]
            
            fixed_opponent_ids.append(opponent_id)
            fixed_opponent_names.append(opponent_name)
            fixed_opponent_difficulties.append(difficulty)
            fixed_is_home.append(is_home)
            
        except (KeyError, FileNotFoundError) as e:
            print(f"Error processing row {idx}: {e}")
            fixed_opponent_ids.append(None)
            fixed_opponent_names.append(None)
            fixed_opponent_difficulties.append(None)
            fixed_is_home.append(None)
    
    df['Opponent ID'] = fixed_opponent_ids
    df['Opponent Name'] = fixed_opponent_names
    df['Opponent Difficulty'] = fixed_opponent_difficulties
    df['Is Home'] = fixed_is_home
    
    return df

In [20]:


def load_fixtures(data_folder='data'):
    gameweeks_per_season = {}
    for season in os.listdir(data_folder):
        matches_per_gameweek = {}
        fixtures_path = os.path.join(data_folder, season, 'fixtures.csv')
        try:
            fixtures_df = pd.read_csv(fixtures_path)
            for _, row in fixtures_df.iterrows():
                event = row['event']
                if event not in matches_per_gameweek:
                    matches_per_gameweek[event] = []
                matches_per_gameweek[event].append(row)
            gameweeks_per_season[season] = matches_per_gameweek
        except FileNotFoundError:
            print(f"Warning: fixtures.csv not found for season {season}")
    return gameweeks_per_season

#def add_opponent_difficulty(df, data_folder='data'):
    """
    Add Opponent Difficulty column by matching against fixtures data.
    
    Parameters:
    df: DataFrame with columns [season, Gameweek, Opponent ID, Is Home]
    data_folder: Path to the data folder containing season subdirectories
    
    Returns:
    DataFrame with new 'Opponent Difficulty' column
    """
    opponent_difficulties = []
    match_dictionary = load_fixtures(data_folder)
    
    for idx, row in df.iterrows():
        season = row['season']
        gameweek = row['Gameweek']
        opponent_id = row['Opponent ID']
        is_home = row['Is Home']
        
        try:
            gameweek_matches = match_dictionary[season][gameweek]
            difficulty = None
            
            # Find the row where opponent matches
            for match in gameweek_matches:
                if is_home:
                    # Looking for team_a = Opponent ID, take team_h_difficulty
                    if match['team_a'] == opponent_id:
                        difficulty = match['team_h_difficulty']
                        break
                else:
                    # Looking for team_h = Opponent ID, take team_a_difficulty
                    if match['team_h'] == opponent_id:
                        difficulty = match['team_a_difficulty']
                        break
            
            opponent_difficulties.append(difficulty)
            
        except (KeyError, FileNotFoundError):
            opponent_difficulties.append(None)
    
    df['Opponent Difficulty'] = opponent_difficulties
    return df

In [22]:
# Apply the fix
print("Fixing opponent data...")
df_fixed = fix_opponent_and_difficulty(df, data_folder='data')

# %%
# Verify the fix for Sokratis
fixed_row = df_fixed[
    (df_fixed['Player Name'] == 'Sokratis Papastathopoulos') & 
    (df_fixed['season'] == '2019-20') & 
    (df_fixed['Gameweek'] == 29)
]

print("\nFixed Training Data Row:")
print(fixed_row[['Player Name', 'Player Team Name', 'season', 'Gameweek', 
                 'Opponent Name', 'Opponent ID', 'Is Home', 'Opponent Difficulty']])


Fixing opponent data...
Error processing row 0: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 1: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 2: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 3: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 4: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 5: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 6: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 7: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 8: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 9: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error processing row 10: [Errno 2] No such file or directory: 'data\\2018-19\\teams.csv'
Error p

KeyboardInterrupt: 

In [ ]:
# Save the corrected data
df_fixed.to_csv('output/training_data_fixed.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Fixed file saved to: output/training_data_fixed.csv")
print(f"📊 Total rows: {len(df_fixed):,}")

In [ ]:
# Add opponent difficulty
#df = add_opponent_difficulty(df, data_folder='data')



DataFrame with Opponent Difficulty:
        A                           Player UUID  Code  \
0  150329  05b195d0-ea16-4d04-bc9b-8567821f271b    12   
1   94783  0ee37250-4bdb-4c84-94f0-afa10ac4a9af    10   
2   36939  107ff853-e67a-4551-abb0-0c6f29d55e44    21   
3   61776  2403603b-153c-4391-a800-f6c4e7536487    18   
4    3977  3003d693-61f9-4157-bf7f-51389503be0c    20   

                 Player Name        Web Name Player Team Name   season  \
0  Sokratis Papastathopoulos        Sokratis          Arsenal  2018-19   
1    Konstantinos Mavropanos      Mavropanos          Arsenal  2018-19   
2              Danny Welbeck         Welbeck          Arsenal  2018-19   
3         Henrikh Mkhitaryan      Mkhitaryan          Arsenal  2018-19   
4     Ainsley Maitland-Niles  Maitland-Niles          Arsenal  2018-19   

   Gameweek  Minutes Played  Goals Scored  ...  Team_Total_Points_GW  \
0         1              90             0  ...                    19   
1         1               0    

In [21]:

# Save the dataframe to CSV
df.to_csv('output/training_data.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ File saved successfully to: output/training_data.csv")
print(f"📊 Total rows saved: {len(df):,}")



✅ File saved successfully to: output/training_data.csv
📊 Total rows saved: 168,972
